In [ ]:
import numpy as np
from copy import deepcopy


class Graph:
    def __init__(self, graph=None, directed=0):
        # Graf przechowywany jako słownik list sąsiedztwa
        if graph is None:
            graph = {}
        self.graph = graph
        self.directed = directed

    @classmethod
    def from_dict(cls, graph):
        return cls(graph)

    @classmethod
    def from_matrix(cls, matrix, vertices=None):
        # Tworzy graf z macierzy sąsiedztwa
        if (vertices is None) or (len(vertices) != len(matrix)):
            vertices = [*range(1, len(matrix) + 1)]
        return cls.from_dict(cls._matrix_to_dict(matrix, vertices))

    @staticmethod
    def _matrix_to_dict(matrix, vertices: list) -> dict:
        # Zamiana macierzy sąsiedztwa na słownik sąsiedztwa
        res_dict = {}
        for i, v in enumerate(vertices):
            neighbours = [vertices[j] for j, edge in enumerate(matrix[i]) if edge]
            res_dict[v] = neighbours
        return res_dict

    def _dict_to_matrix(self, _dict: dict):
        # Zamiana słownika sąsiedztwa na macierz sąsiedztwa
        n = len(_dict)
        vertices = [*_dict.keys()]
        matrix = np.zeros(shape=(n, n), dtype=int)
        for u, v in [
            (vertices.index(u), vertices.index(v))
            for u, row in _dict.items() for v in row
        ]:
            matrix[u][v] += 1
        return matrix

    def vertices(self) -> list:
        return [*self.graph.keys()]

    def matrix(self):
        return self._dict_to_matrix(self.graph)

    def __str__(self):
        # Czytelne wypisanie list sąsiedztwa
        res = ""
        for v in self.graph:
            res += f"{v}:"
            for u in self.graph[v]:
                res += f" {u}"
            res += "\n"
        return res

    def add_vertex(self, vertex):
        if vertex not in self.graph:
            self.graph[vertex] = []

    def add_arc(self, arc):
        # Dodaje łuk skierowany u -> v
        u, v = arc
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)

    def add_edge(self, edge):
        # Dodaje krawędź nieskierowaną u -- v
        u, v = edge
        if u == v:
            raise ValueError("Pętle nie są dopuszczalne!")
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)
        if u not in self.graph[v]:
            self.graph[v].append(u)

    @staticmethod
    def from_edges(filename: str, directed=0):
        # Tworzy graf z pliku zawierającego listę krawędzi
        graph = Graph(directed=directed)
        with open(filename, "r") as file:
            for line in file:
                words = line.strip().split()
                if len(words) == 1:
                    graph.add_vertex(words[0])
                elif len(words) >= 2:
                    if directed:
                        graph.add_arc((words[0], words[1]))
                    else:
                        graph.add_edge((words[0], words[1]))
        return graph

    def Prufer(self):
        # Generuje kod Prüfera drzewa
        tr = deepcopy(self.graph)
        code = []
        for _ in range(len(self.graph) - 2):
            # Szukamy najmniejszego liścia
            for x in sorted(tr):
                if len(tr[x]) == 1:
                    break
            # Do kodu wpisujemy sąsiada tego liścia
            v = tr[x][0]
            code.append(str(v))
            # Usuwamy liść z drzewa
            tr.pop(x)
            tr[v].remove(x)
        return " ".join(code)

    @staticmethod
    def tree_from_Prufer(code: str):
        # Odtwarza drzewo z kodu Prüfera
        tree = Graph()
        clist = [int(x) for x in code.strip().split()]
        n = len(clist) + 2
        vert = [x for x in range(1, n + 1)]

        for v in vert:
            tree.add_vertex(v)

        for _ in range(n - 2):
            # Szukamy najmniejszego numeru, którego nie ma już w kodzie
            for x in vert:
                if x not in clist:
                    break
            v = clist.pop(0)
            tree.add_edge((x, v))
            vert.remove(x)

        # Na końcu zostają dwa wierzchołki, które trzeba połączyć
        tree.add_edge((vert[0], vert[1]))
        return tree

    def preorder(self, v):
        """
        Wypisuje drzewo w porządku preorder, zaczynając od wierzchołka v.
        Uwaga: nie jest sprawdzane, czy graf jest drzewem.
        """
        def dfs(x, parent):
            # PREORDER: najpierw bieżący wierzchołek
            print(x, end=" ")
            for u in sorted(self.graph[x]):
                if u != parent:
                    dfs(u, x)

        dfs(v, None)
        print()

    def postorder(self, v):
        """
        Wypisuje drzewo w porządku postorder, zaczynając od wierzchołka v.
        Uwaga: nie jest sprawdzane, czy graf jest drzewem.
        """
        def dfs(x, parent):
            for u in sorted(self.graph[x]):
                if u != parent:
                    dfs(u, x)
            # POSTORDER: najpierw dzieci, potem bieżący wierzchołek
            print(x, end=" ")

        dfs(v, None)
        print()


tree = Graph.tree_from_Prufer("5 4 3 1")
print(tree)

tree.preorder(5)
tree.postorder(5)


1: 3 6
2: 5
3: 4 1
4: 5 3
5: 2 4
6: 1

5 2 4 3 1 6 
2 6 1 3 4 5 


In [ ]:
from random import random, seed

def random_bipartite_graph(n: int, p: float):
    """
    Tworzy losowy graf dwudzielny o 2n wierzchołkach.
    Pierwsza część to wierzchołki 1, 2, ..., n,
    druga część to wierzchołki n+1, n+2, ..., 2n.
    Każda możliwa krawędź między częściami jest losowana
    niezależnie z prawdopodobieństwem p.
    """
    if not (0 <= p <= 1):
        raise ValueError("p musi należeć do [0, 1]")

    bip_graph = Graph()

    for i in range(1, 2 * n + 1):
        bip_graph.add_vertex(i)

    for i in range(1, n + 1):
        for j in range(n + 1, 2 * n + 1):
            if random() < p:
                bip_graph.add_edge((i, j))

    return bip_graph


seed(2026)
g = random_bipartite_graph(5, 0.4)
print(g)


1: 6 10
2: 6
3: 10
4: 7
5: 8 9
6: 1 2
7: 4
8: 5
9: 5
10: 1 3

